<a href="https://colab.research.google.com/github/gmauricio-toledo/tda/blob/main/notebooks/05-Distancias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gudhi

In [ ]:
!pip install persim

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gudhi
from scipy.spatial.distance import squareform
import seaborn as sns
from persim import wasserstein

np.random.seed(42)

# ============================================
# 1. DATASET BASE: Círculo
# ============================================
n = 100
theta = np.linspace(0, 2*np.pi, n, endpoint=False)
circle = np.column_stack([np.cos(theta), np.sin(theta)])

# ============================================
# 2. PERTURBACIONES DEL CÍRCULO
# ============================================
# Perturbación 1: poco ruido
circle_noise1 = circle + np.random.normal(0, 0.05, circle.shape)

# Perturbación 2: más ruido
circle_noise2 = circle + np.random.normal(0, 0.15, circle.shape)

# Perturbación 3: ruido + menos puntos
circle_noise3 = circle[::2] + np.random.normal(0, 0.1, (n//2, 2))

# ============================================
# 3. DATASET TOPOLÓGICAMENTE DIFERENTE: Dos círculos
# ============================================
theta2 = np.linspace(0, 2*np.pi, n//2, endpoint=False)
circle_small = 0.5 * np.column_stack([np.cos(theta2), np.sin(theta2)]) + np.array([2.5, 0])
two_circles = np.vstack([circle[:n//2], circle_small])
two_circles = two_circles + np.random.normal(0, 0.05, two_circles.shape)

# ============================================
# 4. VISUALIZAR DATASETS
# ============================================
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
datasets = [circle, circle_noise1, circle_noise2, circle_noise3, two_circles]
titles = ['Original', 'Ruido σ=0.05', 'Ruido σ=0.15', 'Menos puntos, ruido σ=0.1', 'Dos círculos']

for ax, data, title in zip(axes, datasets, titles):
    ax.scatter(data[:, 0], data[:, 1], s=20, alpha=0.6)
    ax.set_title(title, fontweight='bold')
    ax.axis('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================
# 5. CALCULAR PERSISTENCIA CON GUDHI
# ============================================
all_datasets = [circle, circle_noise1, circle_noise2, circle_noise3, two_circles]
all_diagrams = []

for data in all_datasets:
    rips = gudhi.RipsComplex(points=data, max_edge_length=2.0)
    st = rips.create_simplex_tree(max_dimension=2)
    persistence = st.persistence()

    # Extraer H1 (ciclos)
    h1 = [(birth, death) for dim, (birth, death) in persistence
          if dim == 1 and death != float('inf')]
    all_diagrams.append(np.array(h1) if len(h1) > 0 else np.array([[0, 0]]))

# ============================================
# 6. VISUALIZAR DIAGRAMAS
# ============================================
fig, axes = plt.subplots(1, 5, figsize=(18, 3))

for ax, dgm, title in zip(axes, all_diagrams, titles):
    if len(dgm) > 0 and dgm.shape[0] > 0:
        ax.scatter(dgm[:, 0], dgm[:, 1], s=50, alpha=0.7)
        max_val = max(dgm[:, 1].max(), dgm[:, 0].max()) if len(dgm) > 0 else 1
        ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3)
    ax.set_title(f'H₁: {title}', fontweight='bold')
    ax.set_xlabel('Birth')
    ax.set_ylabel('Death')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================
# 7. CALCULAR MATRIZ DE DISTANCIAS WASSERSTEIN
# ============================================
n_datasets = len(all_diagrams)


n_datasets = len(all_diagrams)
wasserstein_matrix = np.zeros((n_datasets, n_datasets))

for i in range(n_datasets):
    for j in range(i, n_datasets):
        dist = wasserstein(all_diagrams[i], all_diagrams[j])
        wasserstein_matrix[i, j] = dist
        wasserstein_matrix[j, i] = dist

# for i in range(n_datasets):
#     for j in range(i, n_datasets):
#         dist = gudhi.wasserstein.wasserstein(all_diagrams[i], all_diagrams[j], order=1)
#         wasserstein_matrix[i, j] = dist
#         wasserstein_matrix[j, i] = dist

# ============================================
# 8. HEATMAP DE DISTANCIAS
# ============================================
plt.figure(figsize=(8, 6))
sns.heatmap(wasserstein_matrix,
            annot=True,
            fmt='.3f',
            cmap='YlOrRd',
            xticklabels=titles,
            yticklabels=titles,
            cbar_kws={'label': 'Wasserstein Distance'})
plt.title('Matriz de Distancias de Wasserstein (H₁)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

# ============================================
# 9. INTERPRETACIÓN
# ============================================
print("Distancias entre círculos perturbados:")
print(f"  Original vs Ruido σ=0.05:  {wasserstein_matrix[0, 1]:.3f}")
print(f"  Original vs Ruido σ=0.15:  {wasserstein_matrix[0, 2]:.3f}")
print(f"  Original vs Menos puntos, ruido σ=0.1:  {wasserstein_matrix[0, 3]:.3f}")
print(f"\nDistancia a topología diferente:")
print(f"  Original vs Dos círculos:  {wasserstein_matrix[0, 4]:.3f}")